In [1]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font

# Carregar as bases
baseclientes = pd.read_excel('baseclientes.xlsx')
basesac = pd.read_excel('basesac.xlsx')

# Juntar pelo Telefone
dados_combinados = pd.merge(
    basesac[['Telefone', 'Nome', 'Cód', 'Plano', 'Clínica', 'Serviço']],
    baseclientes[['Telefone', 'Responsável']],
    on='Telefone',
    how='left'
)

# Agrupar por cliente
agrupado = dados_combinados.groupby(
    ['Telefone', 'Responsável', 'Nome', 'Cód', 'Plano', 'Clínica']
)

blocos = []
soma_total_geral = 0  # Contador total de serviços únicos

for chave, grupo in agrupado:
    servicos_unicos = grupo['Serviço'].dropna().unique()
    total_servicos_unicos = len(servicos_unicos)
    soma_total_geral += total_servicos_unicos
    blocos.append((total_servicos_unicos, chave, servicos_unicos))

# Ordenar por número de serviços únicos (descrescente)
blocos.sort(reverse=True, key=lambda x: x[0])

# Criar Excel
wb = Workbook()
ws = wb.active
ws.title = "Atendimentos do dia"
fonte_negrito = Font(bold=True)
linha_atual = 1

# Função para formatar telefone
def formatar_telefone(numero):
    num_str = str(numero)
    if len(num_str) == 11:
        return f"({num_str[:2]}){num_str[2:7]}-{num_str[7:]}"
    return num_str

# Preencher a planilha
for total_servicos_unicos, (telefone, responsavel, nome, cod, plano, clinica), servicos_unicos in blocos:
    ws.cell(row=linha_atual, column=1, value=cod)
    linha_atual += 1

    ws.cell(row=linha_atual, column=1, value=responsavel)
    ws.cell(row=linha_atual, column=2, value=total_servicos_unicos)
    linha_atual += 1

    ws.cell(row=linha_atual, column=1, value=nome)
    linha_atual += 1

    ws.cell(row=linha_atual, column=1, value=formatar_telefone(telefone))
    linha_atual += 1

    celula_plano = ws.cell(row=linha_atual, column=1, value=plano)
    celula_plano.font = fonte_negrito
    linha_atual += 1

    ws.cell(row=linha_atual, column=1, value=clinica)
    linha_atual += 1

    for servico in servicos_unicos:
        ws.cell(row=linha_atual, column=1, value=servico)
        linha_atual += 1

    linha_atual += 1  # Espaço entre clientes

# Soma total no final
ws.cell(row=linha_atual, column=1, value="TOTAL DE SERVIÇOS ÚNICOS:")
ws.cell(row=linha_atual, column=2, value=soma_total_geral)
ws.cell(row=linha_atual, column=1).font = fonte_negrito
ws.cell(row=linha_atual, column=2).font = fonte_negrito

# Ajustar larguras
ws.column_dimensions['A'].width = 50
ws.column_dimensions['B'].width = 15

# Salvar
wb.save("Atendimentos_do_dia.xlsx")
print("Relatório gerado com sucesso: Atendimentos_do_dia.xlsx")

Relatório gerado com sucesso: Atendimentos_do_dia.xlsx


In [2]:
import pandas as pd
from openpyxl import load_workbook

def gerar_arquivo_clientes():
    try:
        # Carregar o arquivo gerado anteriormente
        arquivo_original = "Atendimentos_do_dia.xlsx"
        wb = load_workbook(arquivo_original)
        ws = wb.active

        # Listas para armazenar os dados
        nomes = []
        animais = []
        telefones = []

        # Percorrer as linhas do arquivo original
        for i, row in enumerate(ws.iter_rows(values_only=True), 1):
            # Procurar linhas com código (indicam início de um novo cliente)
            if isinstance(row[0], (int, float, str)) and str(row[0]).isdigit():
                # A linha abaixo do código é o Responsável, então pulamos
                # A linha seguinte (i+2) é o Nome
                nome = ws.cell(row=i+1, column=1).value
                # A linha abaixo do Nome (i+3) é o Animal
                animal = ws.cell(row=i+2, column=1).value
                # A linha abaixo do Nome (i+3) é o Animal
                telefone = ws.cell(row=i+3, column=1).value

                if nome and animal:  # Só adiciona se ambos existirem
                    nomes.append(nome)
                    animais.append(animal)
                    telefones.append(telefone)

        # Criar DataFrame com os dados extraídos
        dados_clientes = pd.DataFrame({
            'Nome': nomes,
            'Animal': animais,
            'Telefone': telefones
        })

        # Remover duplicatas (caso um mesmo cliente apareça mais de uma vez)
        dados_clientes = dados_clientes.drop_duplicates()

        # Criar novo arquivo Excel
        with pd.ExcelWriter('clientes.xlsx') as writer:
            dados_clientes.to_excel(writer, sheet_name='Clientes', index=False)

            # Ajustar o tamanho das colunas
            worksheet = writer.sheets['Clientes']
            worksheet.column_dimensions['A'].width = 30  # Coluna Nome
            worksheet.column_dimensions['B'].width = 30  # Coluna Animal
            worksheet.column_dimensions['C'].width = 30  # Coluna Animal

        print("\nArquivo 'clientes.xlsx' gerado com sucesso!")
        print(f"Total de clientes cadastrados: {len(dados_clientes)}")

    except Exception as e:
        print("\nOcorreu um erro ao gerar o arquivo de clientes:")
        print(str(e))

# Executar a função
gerar_arquivo_clientes()


Arquivo 'clientes.xlsx' gerado com sucesso!
Total de clientes cadastrados: 59


In [3]:
import pandas as pd

# Base principal (mensagens)
df_clientes = pd.read_excel('clientes.xlsx')

# Base auxiliar (apenas sexo do animal)
df_sexo = pd.read_excel('/content/baseclientes.xlsx')

# Criar um dicionário: Animal → Sexo
mapa_sexo = dict(
    zip(
        df_sexo['Animal'].astype(str).str.strip().str.lower(),
        df_sexo['Sexo']
    )
)

def artigo_por_sexo(sexo):
    if pd.isna(sexo):
        return 'do(a)'
    sexo = str(sexo).strip().lower()
    if sexo in ['f', 'feminino', 'fêmea', 'femea']:
        return 'da'
    else:
        return 'do'

mensagens = []

for _, row in df_clientes.iterrows():
    # Cliente → apenas primeiro nome
    cliente_completo = str(row['Nome']).strip().title()
    primeiro_nome = cliente_completo.split()[0]

    # Animal
    animal = str(row['Animal']).strip().title()
    animal_key = animal.lower()

    # Telefone
    telefone = str(row['Telefone']).strip()

    # Buscar sexo SOMENTE no baseclientes.xlsx
    sexo = mapa_sexo.get(animal_key)
    artigo = artigo_por_sexo(sexo)

    mensagem = f"""{telefone}
Olá, {primeiro_nome}! Aqui é do Serviço de Satisfação do Cliente do Apaixonados Saúde Pet, o plano {artigo} {animal}! Gostaríamos de saber como foi o seu último atendimento.

Sua opinião é muito importante para nós!

- Você gostou do atendimento prestado na Clínica?
- Assinou a guia de solicitação de procedimentos e exames?"""

    mensagens.append(mensagem)

# Salvar resultado final
with open('mensagens_automatizadas.txt', 'w', encoding='utf-8') as f:
    for msg in mensagens:
        f.write(msg + '\n\n')
